# A.1 給幾個例子能改變回答嗎？


你對模型說「請把3轉成另一個數字」，它不知道你想翻倍、平方還是換成代碼。如果先給「1→2、2→4」兩個例子，至少提供了本次想要的規律與輸出格式。模型權重完全不動，輸入卻已不同：它現在可以讀例子再回答。這種靠當前前文提供任務線索的現象叫in-context learning（上下文學習）。前置是[提示怎樣改變回答](https://birdhackor.github.io/tiny-perceptron-vlm/8.2.html)，context就是這次模型能看到的文字與其他輸入。

沒有示例、只給任務叫zero-shot；多給幾個示例叫few-shot。「shot」在這裡數的是提示中的例子，不是訓練步數。若把例子換成「1→3、2→6」，我們預期同樣的3可能轉成9而不是6；答案差異來自本次輸入條件，不代表權重記住了一個新的永久法則。對有限例子還可能有多種解讀，所以真實回答不能只靠人的推測。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
question = "請依本次規則回答：3→?"
examples = ["1→2", "2→4"]
zero = question
few = "示例：" + "；".join(examples) + "。\n" + question
changed = "示例：1→3；2→6。\n" + question
print("無例子：", zero)
print("兩例子：", few)
print("換規則：", changed)

輸入是一個問題與兩條示例字串，`join`用分號把例子接起來，`\n`換行後才放真正問題。輸出三份提示，前兩份問題相同，第三份只替換示例規則。這個程式沒有載入模型，也沒有證明模型輸出6或9；它將要比較的輸入清楚展開，讓我們先檢查「改了什麼」。不要用一個自己寫的翻倍函數代替模型回答，否則測到的是規則程式，不是上下文學習。

若用可用模型做比較，應保持同一權重、同樣生成設定與問題，記錄原始提示和回答，再換示例。示例的排序、標籤、數值分布都可能影響結果；也應包含新數值或新組合，避免只考「複製提示裡已經出現的答案」。[訓練對話SFT](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)則會更新權重，是另一件事。把例子從prompt移除，模型不會因這次讀過它們就必然繼續遵循。

練習只把examples兩項順序交換，保留question不變。先預測打印的規則仍是兩個翻倍示例，但排列不同；執行核對。如果之後做真實模型實驗，先寫下你預期答案是否應相同，再保存兩份實際回答。這能分清「人在例子裡看見相同規則」和「模型確實對順序穩定」兩個觀察。
